# 🎨 Director-X Image Server — Colab Edition

Generates *cinematic still images* on Colab's free T4 GPU for Director-X's motion pipeline.

**How it works:** Director-X sends scene descriptions → this generates stunning images → Director-X adds cinematic motion (Ken Burns zoom/pan) + voice narration → finished video.

**~5-10 seconds per image. No crashes. Proven on free T4.**

---

## Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    print(f"\n✅ {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_mem / 1024**3:.0f} GB")
else:
    print("\n❌ No GPU! Runtime → Change runtime type → T4 GPU")

## Step 1: Install packages

In [ ]:
!pip install -q flask flask-cors pyngrok Pillow
!pip install -q diffusers[torch] transformers accelerate safetensors
print("\n✅ Done")

## Step 2: ngrok token
Free at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

if not NGROK_AUTH_TOKEN:
    print("⚠️  Paste your ngrok token above!")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok ready")

## Step 3: Load model
First run downloads ~6 GB. Cached after that (~1 min reload).

In [ ]:
import torch, gc

dtype = torch.float16
device = "cuda"
torch.cuda.empty_cache()
gc.collect()

img_pipe = None
active_model = None

# ─── Try SDXL first (best quality) ───
try:
    print("🎨 Loading SDXL 1.0 (best image quality)...")
    from diffusers import StableDiffusionXLPipeline

    img_pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=dtype,
        variant="fp16",
        use_safetensors=True,
    )
    img_pipe.enable_sequential_cpu_offload()
    try:
        img_pipe.enable_xformers_memory_efficient_attention()
        print("   ✅ xformers enabled")
    except Exception:
        pass

    # Quick test
    print("   Testing VRAM fit...")
    with torch.inference_mode():
        _t = img_pipe(prompt="test", num_inference_steps=1, width=512, height=512).images[0]
    del _t
    torch.cuda.empty_cache()
    gc.collect()

    active_model = "SDXL-1.0"
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"\n✅ SDXL loaded! Peak VRAM: {peak:.1f} GB")
    print("   Resolution: 1024×576 (16:9) | ~5-10 sec/image")

except Exception as e:
    print(f"   ⚠️ SDXL failed: {str(e)[:100]}")
    print("   Falling back to SD 1.5...")
    del img_pipe
    img_pipe = None
    torch.cuda.empty_cache()
    gc.collect()

# ─── Fallback: SD 1.5 (lighter, still good) ───
if img_pipe is None:
    try:
        from diffusers import StableDiffusionPipeline

        img_pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=dtype,
            variant="fp16",
            use_safetensors=True,
        )
        img_pipe.enable_sequential_cpu_offload()

        active_model = "SD-1.5"
        print(f"\n✅ Stable Diffusion 1.5 loaded!")

    except Exception as e:
        raise RuntimeError(f"No image model could load: {e}")

print(f"\n📋 Active model: {active_model}")
print(f"   Ready to generate scene images for Director-X!")

## Step 4: Quick test (optional)

In [ ]:
import time

print(f"🎨 Testing {active_model}...")
torch.cuda.empty_cache(); gc.collect()

start = time.time()
with torch.inference_mode():
    test_img = img_pipe(
        prompt="A lone sheriff stands at the end of a dusty frontier town street at golden hour, dramatic shadows, cinematic wide shot",
        negative_prompt="blurry, low quality, cartoon, anime",
        width=1024, height=576,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=torch.Generator(device="cuda").manual_seed(42),
    ).images[0]

test_img.save("/content/test.png")
del test_img; torch.cuda.empty_cache(); gc.collect()

elapsed = time.time() - start
print(f"\n✅ {elapsed:.0f}s — saved to /content/test.png")

from IPython.display import display, Image as IPyImage
display(IPyImage(filename="/content/test.png", width=640))

## Step 5: Start server 🚀
Copy the URL → paste into Director-X settings.

In [ ]:
import os, uuid, json, time, threading, gc, io, base64
import torch
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
from collections import OrderedDict
from PIL import Image

app = Flask(__name__)
CORS(app)

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

jobs = OrderedDict()
MAX_JOBS = 200
gen_queue = []
gen_lock = threading.Lock()
is_generating = False

ASPECT_PRESETS = {
    "16:9":  (1024, 576),
    "9:16":  (576, 1024),
    "1:1":   (768, 768),
    "4:3":   (896, 672),
    "4:5":   (640, 800),
}

STYLE_SUFFIXES = {
    "cinematic": ", cinematic lighting, dramatic shadows, film grain, 35mm photography, depth of field, volumetric fog",
    "western":   ", western frontier, dusty atmosphere, golden hour, dramatic sky, Sergio Leone style",
    "noir":      ", film noir, high contrast, dramatic shadows, moody lighting, rain-slicked streets",
    "fantasy":   ", epic fantasy, dramatic lighting, detailed environment, matte painting style",
    "pirate":    ", age of sail, dramatic ocean, candlelit, weathered wood, nautical atmosphere",
    "warfare":   ", battlefield atmosphere, smoke and fire, dramatic sky, gritty realism",
    "urban":     ", urban photography, street level, neon reflections, cinematic city atmosphere",
    "frozen":    ", frozen landscape, ice and snow, cold blue tones, breath visible, harsh winter",
    "none":      "",
}

def generate_image(job_id, prompt, aspect_ratio="16:9", style="cinematic", seed=-1, negative_prompt=None):
    global is_generating
    try:
        is_generating = True
        jobs[job_id]["status"] = "generating"
        torch.cuda.empty_cache()
        gc.collect()

        width, height = ASPECT_PRESETS.get(aspect_ratio, ASPECT_PRESETS["16:9"])
        full_prompt = prompt + STYLE_SUFFIXES.get(style, STYLE_SUFFIXES["cinematic"])

        neg = negative_prompt or "blurry, low quality, distorted, watermark, text, logo, oversaturated, cartoon, anime, 3d render"

        gen_kwargs = dict(
            prompt=full_prompt,
            negative_prompt=neg,
            width=width,
            height=height,
            num_inference_steps=30,
            guidance_scale=7.5,
        )
        if seed >= 0:
            gen_kwargs["generator"] = torch.Generator(device="cuda").manual_seed(seed)

        with torch.inference_mode():
            image = img_pipe(**gen_kwargs).images[0]

        output_path = os.path.join(OUTPUT_DIR, f"{job_id}.png")
        image.save(output_path, quality=95)

        # Also save a base64 thumbnail for quick preview
        thumb = image.copy()
        thumb.thumbnail((384, 384))
        buf = io.BytesIO()
        thumb.save(buf, format="JPEG", quality=80)
        thumb_b64 = base64.b64encode(buf.getvalue()).decode()

        del image, thumb
        torch.cuda.empty_cache()
        gc.collect()

        jobs[job_id]["status"] = "completed"
        jobs[job_id]["image_path"] = output_path
        jobs[job_id]["thumbnail"] = thumb_b64
        print(f"\u2705 Job {job_id[:8]} done: {prompt[:50]}...")

    except torch.cuda.OutOfMemoryError:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = "GPU out of memory"
        torch.cuda.empty_cache()
        gc.collect()
    except Exception as e:
        jobs[job_id]["status"] = "failed"
        jobs[job_id]["error"] = str(e)
        print(f"\u274c Job {job_id[:8]} failed: {e}")
    finally:
        is_generating = False
        torch.cuda.empty_cache()
        gc.collect()
        process_queue()

def process_queue():
    global is_generating
    with gen_lock:
        if is_generating or not gen_queue:
            return
        job = gen_queue.pop(0)
    thread = threading.Thread(target=generate_image, kwargs=job)
    thread.start()

@app.route("/api/health", methods=["GET"])
def health():
    vram = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3 if torch.cuda.is_available() else 0
    return jsonify({
        "status": "ok",
        "provider": "colab-image",
        "model": active_model,
        "type": "image",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_used_gb": round(vram, 1),
        "vram_total_gb": round(total, 1),
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "supported_aspects": list(ASPECT_PRESETS.keys()),
        "supported_styles": list(STYLE_SUFFIXES.keys()),
    })

@app.route("/api/image/generate", methods=["POST"])
def submit_image():
    data = request.json or {}
    prompt = data.get("prompt", "").strip()
    if not prompt:
        return jsonify({"error": "prompt is required"}), 400

    job_id = str(uuid.uuid4())
    aspect = data.get("aspect_ratio", data.get("aspectRatio", "16:9"))
    style = data.get("style", "cinematic")
    seed = int(data.get("seed", -1))
    neg = data.get("negative_prompt")

    jobs[job_id] = {
        "status": "queued",
        "prompt": prompt[:200],
        "aspect_ratio": aspect,
        "style": style,
        "created": time.time(),
        "image_path": None,
        "thumbnail": None,
        "error": None,
    }

    while len(jobs) > MAX_JOBS:
        old_id, old_job = jobs.popitem(last=False)
        if old_job.get("image_path") and os.path.exists(old_job["image_path"]):
            os.remove(old_job["image_path"])

    gen_queue.append({
        "job_id": job_id,
        "prompt": prompt,
        "aspect_ratio": aspect,
        "style": style,
        "seed": seed,
        "negative_prompt": neg,
    })
    process_queue()

    return jsonify({
        "requestId": job_id,
        "statusUrl": f"/api/image/status/{job_id}",
        "provider": "colab-image",
        "model": active_model,
        "queue_position": len(gen_queue),
    })

@app.route("/api/image/status/<job_id>", methods=["GET"])
def image_status(job_id):
    if job_id not in jobs:
        return jsonify({"error": "Job not found"}), 404
    job = jobs[job_id]
    result = {
        "status": job["status"].upper(),
        "prompt": job["prompt"],
        "aspect_ratio": job.get("aspect_ratio"),
        "style": job.get("style"),
    }
    if job["status"] == "completed":
        result["imageUrl"] = f"/api/image/download/{job_id}"
        if job.get("thumbnail"):
            result["thumbnail"] = f"data:image/jpeg;base64,{job['thumbnail']}"
    elif job["status"] == "failed":
        result["error"] = job.get("error", "Unknown error")
    elif job["status"] == "queued":
        pos = next((i for i, j in enumerate(gen_queue) if j["job_id"] == job_id), -1)
        result["queue_position"] = pos + 1 if pos >= 0 else 0
    return jsonify(result)

@app.route("/api/image/download/<job_id>", methods=["GET"])
def download_image(job_id):
    if job_id not in jobs or not jobs[job_id].get("image_path"):
        return jsonify({"error": "Image not found"}), 404
    return send_file(jobs[job_id]["image_path"], mimetype="image/png")

@app.route("/api/image/batch", methods=["POST"])
def batch_images():
    """Submit multiple image prompts at once (for full scene generation)."""
    data = request.json or {}
    scenes = data.get("scenes", [])
    if not scenes:
        return jsonify({"error": "scenes array required"}), 400

    results = []
    for scene in scenes[:50]:
        prompt = scene.get("prompt", "").strip()
        if not prompt:
            continue
        job_id = str(uuid.uuid4())
        aspect = scene.get("aspect_ratio", data.get("aspect_ratio", "16:9"))
        style = scene.get("style", data.get("style", "cinematic"))
        seed = int(scene.get("seed", -1))

        jobs[job_id] = {
            "status": "queued",
            "prompt": prompt[:200],
            "aspect_ratio": aspect,
            "style": style,
            "created": time.time(),
            "image_path": None,
            "thumbnail": None,
            "error": None,
        }
        gen_queue.append({
            "job_id": job_id,
            "prompt": prompt,
            "aspect_ratio": aspect,
            "style": style,
            "seed": seed,
        })
        results.append({
            "requestId": job_id,
            "statusUrl": f"/api/image/status/{job_id}",
            "prompt": prompt[:80],
        })

    process_queue()
    return jsonify({
        "submitted": len(results),
        "jobs": results,
    })

@app.route("/api/queue", methods=["GET"])
def queue_info():
    return jsonify({
        "queue_length": len(gen_queue),
        "is_generating": is_generating,
        "active_model": active_model,
        "completed": sum(1 for j in jobs.values() if j["status"] == "completed"),
        "failed": sum(1 for j in jobs.values() if j["status"] == "failed"),
        "recent_jobs": [
            {"id": jid, "status": j["status"], "prompt": j["prompt"][:50]}
            for jid, j in list(jobs.items())[-10:]
        ],
    })

port = 5000
public_url = ngrok.connect(port)

print("\n" + "=" * 60)
print("\U0001f3a8 DIRECTOR-X IMAGE SERVER IS RUNNING!")
print("=" * 60)
print(f"\n\U0001f916 Model: {active_model}")
print(f"\n\U0001f310 Public URL: {public_url}")
print(f"\n\U0001f4cb Paste into Director-X \u2192 Settings \u2192 Image Server URL:")
print(f"   {public_url}")
print(f"\n\U0001f527 Health: {public_url}/api/health")
print("\n\u26a1 Aspects: 16:9 (YT), 9:16 (TikTok/Reels), 1:1 (IG), 4:5 (FB)")
print("\U0001f3a8 Styles: cinematic, western, noir, fantasy, pirate, warfare, urban, frozen")
print("\n\u23f3 ~5-10 seconds per image. Keep this running!")
print("=" * 60)

app.run(port=port)

---
## 🎨 Styles
| Style | Best for |
|-------|----------|
| `cinematic` | Default — dramatic lighting, film grain |
| `western` | Hell on Wheels / The Kansas City Line |
| `frozen` | The Frozen Front / Game of Thrones |
| `pirate` | No Quarter / Black Sails |
| `noir` | The Departed / Atlantic Command |
| `warfare` | Texas Rising / The Wall |
| `urban` | We Own This City / Eastern Front |
| `fantasy` | The Audacity / The Pacific Coast |

## API
```
POST /api/image/generate   { "prompt": "...", "aspect_ratio": "16:9", "style": "western" }
POST /api/image/batch       { "scenes": [{ "prompt": "...", "style": "..." }, ...] }
GET  /api/image/status/{id} → { status, imageUrl, thumbnail }
GET  /api/image/download/{id} → PNG
GET  /api/health | /api/queue
```
